In [1]:
from pathlib import Path
import pandas as pd

file = Path('../Data/train_data.csv')
df = pd.read_csv(file)

print(df)


        event_id  time_to_tca  mission_id       risk  max_risk_estimate  \
0              0     1.566798           5 -10.204955          -7.834756   
1              0     1.207494           5 -10.355758          -7.848937   
2              0     0.952193           5 -10.345631          -7.847406   
3              0     0.579669           5 -10.337809          -7.845880   
4              0     0.257806           5 -10.391260          -7.852942   
...          ...          ...         ...        ...                ...   
162629     13153     3.029751           1  -7.108630          -5.142668   
162630     13153     2.799253           1  -7.070070          -5.137869   
162631     13153     2.385399           1  -7.066209          -5.137034   
162632     13153     2.043339           1  -7.028307          -5.131297   
162633     13153     1.618095           1  -7.172372          -5.152181   

        max_risk_scaling  miss_distance  relative_speed  relative_position_r  \
0               8.6

In [3]:
final = df.loc[df.groupby('event_id')['time_to_tca'].idxmin()]
(final['risk'] >= -6).sum()    

danger = final.set_index('event_id')['risk'] >= -6
final[final['risk'] >= -6]['event_id'].head()


early = df[df['time_to_tca'] >= 3]
early_risk = early.loc[early.groupby('event_id')['time_to_tca'].idxmin()].set_index('event_id')['risk']

cmp = pd.DataFrame({'early_risk': early_risk, 'ended_dangerous': danger}).dropna()
print(cmp.groupby('ended_dangerous')['early_risk'].describe())

looked_dangerous = cmp['early_risk'] >= -6
print('Looked dangerous at 3+ days:', looked_dangerous.sum())
print('...and ended dangerous:', (looked_dangerous & cmp['ended_dangerous']).sum())
print('Ended dangerous:', cmp['ended_dangerous'].sum())


                   count       mean       std   min        25%        50%  \
ended_dangerous                                                             
False            10971.0 -18.394638  9.746377 -30.0 -30.000000 -15.022780   
True               302.0  -5.689852  2.236619 -30.0  -5.809949  -5.568235   

                      75%       max  
ended_dangerous                      
False           -8.773014 -2.924818  
True            -5.158219 -1.684660  
Looked dangerous at 3+ days: 705
...and ended dangerous: 276
Ended dangerous: 302


In [6]:
observed = df[df['time_to_tca'] >= 2]
print(last_2)

        event_id  time_to_tca  mission_id       risk  max_risk_estimate  \
5              1     6.530455           5  -7.561299          -7.254301   
6              1     5.561646           5  -9.315693          -7.468904   
7              1     5.226504           5  -7.422508          -7.051001   
8              1     3.570013           5  -9.248105          -7.327533   
9              2     6.983474           2 -10.816161          -6.601713   
...          ...          ...         ...        ...                ...   
162628     13153     3.408859           1  -7.080451          -5.140501   
162629     13153     3.029751           1  -7.108630          -5.142668   
162630     13153     2.799253           1  -7.070070          -5.137869   
162631     13153     2.385399           1  -7.066209          -5.137034   
162632     13153     2.043339           1  -7.028307          -5.131297   

        max_risk_scaling  miss_distance  relative_speed  relative_position_r  \
5               2.7

In [11]:
forecast = observed.loc[observed.groupby('event_id')['time_to_tca'].idxmin()]
print(forecast)

        event_id  time_to_tca  mission_id       risk  max_risk_estimate  \
8              1     3.570013           5  -9.248105          -7.327533   
22             2     2.340627           2 -30.000000          -6.266241   
43             3     2.278941          19 -30.000000          -7.320481   
52             4     3.066467          19 -30.000000          -7.661743   
72             5     2.103772           5 -13.100070          -4.878440   
...          ...          ...         ...        ...                ...   
162589     13149     5.069389           6  -9.162854          -6.599808   
162591     13150     6.322295           9 -30.000000          -7.004628   
162599     13151     5.093112           5  -9.791559          -6.734710   
162614     13152     2.128300          15 -30.000000          -7.103143   
162632     13153     2.043339           1  -7.028307          -5.131297   

        max_risk_scaling  miss_distance  relative_speed  relative_position_r  \
8               7.4

In [17]:
pd.merge(final,forecast, on="event_id", )

,event_id,time_to_tca_x,mission_id_x,risk_x,max_risk_estimate_x,max_risk_scaling_x,miss_distance_x,relative_speed_x,relative_position_r_x,relative_position_t_x,...,t_sigma_rdot_y,c_sigma_rdot_y,t_sigma_tdot_y,c_sigma_tdot_y,t_sigma_ndot_y,c_sigma_ndot_y,F10_y,F3M_y,SSN_y,AP_y
0,1,3.570013,5,-9.248105,-7.327533,7.425994,26899.0,3434.0,-82.0,-26067.0,...,0.124802,30.242768,0.005883,0.174956,0.003408,0.058311,71.0,87.0,21.0,5.0
1,2,0.401947,2,-30.000000,-7.449283,37296.168207,18708.0,14347.0,-717.9,-5159.0,...,0.033330,3.039035,0.004753,0.053199,0.003115,0.194275,69.0,77.0,11.0,6.0
2,3,0.283061,19,-30.000000,-8.439735,45859.719269,23861.0,13574.0,51.8,10055.9,...,0.095417,3.976678,0.008990,0.056902,0.007782,0.160887,68.0,70.0,0.0,7.0
3,4,0.273166,19,-27.650917,-7.819587,50.584357,23080.0,12093.0,187.1,-13552.1,...,0.081852,2.058484,0.007439,0.054042,0.005327,0.076470,84.0,86.0,30.0,36.0
4,5,0.084142,5,-30.000000,-5.238523,387.328405,329.0,2001.0,-12.6,-326.2,...,0.036157,0.037449,0.005818,0.005165,0.004303,0.004416,70.0,77.0,11.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11937,13149,5.069389,6,-9.162854,-6.599808,9.114065,8239.0,12150.0,121.3,-6419.3,...,3.250008,46.734056,0.007076,1.359989,0.001840,0.072661,68.0,70.0,12.0,17.0
11938,13150,0.378560,9,-30.000000,-6.340559,225065.730452,11963.0,14953.0,-1037.8,1305.3,...,0.313001,19.781598,0.001430,0.070820,0.002989,0.027618,69.0,69.0,0.0,5.0
11939,13151,5.093112,5,-9.791559,-6.734710,10.393204,34897.0,11876.0,87.1,-21516.7,...,0.210887,15.310877,0.008980,0.026154,0.004643,0.130341,69.0,70.0,0.0,4.0
11940,13152,0.144192,15,-30.000000,-7.394695,19916.402793,25222.0,1156.0,-1204.7,-25151.7,...,0.031848,0.554859,0.001393,0.041585,0.001586,0.102717,71.0,72.0,15.0,4.0


In [26]:
result = pd.merge(final[['event_id', 'risk']],forecast[['event_id', 'risk']],on='event_id')
print(result)

       event_id     risk_x     risk_y
0             1  -9.248105  -9.248105
1             2 -30.000000 -30.000000
2             3 -30.000000 -30.000000
3             4 -27.650917 -30.000000
4             5 -30.000000 -13.100070
...         ...        ...        ...
11937     13149  -9.162854  -9.162854
11938     13150 -30.000000 -30.000000
11939     13151  -9.791559  -9.791559
11940     13152 -30.000000 -30.000000
11941     13153  -7.172372  -7.028307

[11942 rows x 3 columns]


In [27]:
result['actual_danger'] = result['risk_x'] >= -6
result['pred_danger'] = result['risk_y'] >= -6

caught = ((result['pred_danger']) & (result['actual_danger'])).sum()
missed = ((~result['pred_danger']) & (result['actual_danger'])).sum()
false_alarm = ((result['pred_danger']) & (~result['actual_danger'])).sum()
all_clear = ((~result['pred_danger']) & (~result['actual_danger'])).sum()

print("Rows:", len(result))
print("Caught:", caught)
print("Missed:", missed)
print("False alarm:", false_alarm)
print("All-clear:", all_clear)

Rows: 11942
Caught: 300
Missed: 23
False alarm: 269
All-clear: 11350


In [25]:
print(result)

       event_id     risk_x     risk_y  actual_danger  pred_danger
0             1  -9.248105  -9.248105          False        False
1             2 -30.000000 -30.000000          False        False
2             3 -30.000000 -30.000000          False        False
3             4 -27.650917 -30.000000          False        False
4             5 -30.000000 -13.100070          False        False
...         ...        ...        ...            ...          ...
11937     13149  -9.162854  -9.162854          False        False
11938     13150 -30.000000 -30.000000          False        False
11939     13151  -9.791559  -9.791559          False        False
11940     13152 -30.000000 -30.000000          False        False
11941     13153  -7.172372  -7.028307          False        False

[11942 rows x 5 columns]
